# LOCO Efficient Detector — Colab Runner

This notebook keeps the dataset and checkpoints in Google Drive, runs repository checks, and resumes interrupted training automatically. Before starting, select **Runtime → Change runtime type → T4 GPU** and add a Colab secret named `GITHUB_TOKEN` with read access to the private repository.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

assert torch.cuda.is_available(), 'Enable a GPU under Runtime > Change runtime type'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GB')

## Mount persistent storage and clone the private repository

In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive')
STORAGE = Path('/content/drive/MyDrive/inmindAcademyDetector')
DATA_ROOT = STORAGE / 'dataset'
RUNS_ROOT = STORAGE / 'runs'
CONFIG_ROOT = STORAGE / 'configs'
for directory in (STORAGE, RUNS_ROOT, CONFIG_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

REPO_URL = 'https://github.com/JonathanAchkouti/inmindAcademyDetector.git'
BRANCH = 'codex/loco-efficient-detector'
REPO_DIR = Path('/content/inmindAcademyDetector')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add a GITHUB_TOKEN secret with access to the private repository')
git_env = os.environ.copy()
git_env.update({
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.extraHeader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: bearer {token}',
})
if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, env=git_env, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, env=git_env, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], env=git_env, check=True)
os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

## Install only the packages missing from Colab

In [ ]:
%pip install -q einops "torchmetrics[detection]" pyyaml wandb ruff pytest
import torchvision
print('Torchvision:', torchvision.__version__)

## Verify the code before downloading data

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'ruff', 'format', '--check', '.'], check=True)
subprocess.run([sys.executable, '-m', 'ruff', 'check', '.'], check=True)

## Download LOCO once to Google Drive

The script skips this step automatically in future sessions.

In [ ]:
subprocess.run(['bash', 'scripts/download_loco.sh', str(DATA_ROOT)], check=True)

## Prepare a persistent experiment configuration

In [ ]:
import yaml
from utils.config import load_config

def prepare_config(source: str) -> Path:
    config = load_config(Path(source))
    run_name = Path(config['output_dir']).name
    config['data']['raw_dir'] = str(DATA_ROOT)
    config['output_dir'] = str(RUNS_ROOT / run_name)
    destination = CONFIG_ROOT / f'{run_name}.yaml'
    destination.write_text(yaml.safe_dump(config, sort_keys=False))
    return destination

SELECTED_EXPERIMENT = 'configs/e03_p3_tuned_512.yaml'
CONFIG_PATH = prepare_config(SELECTED_EXPERIMENT)
print(CONFIG_PATH.read_text())

## Smoke-train before spending GPU hours

In [ ]:
subprocess.run([sys.executable, 'train.py', '--config', str(CONFIG_PATH), '--smoke'], check=True)

## Train or resume the selected experiment

Re-running this cell resumes from `last.pt` in Google Drive.

In [ ]:
subprocess.run([sys.executable, 'train.py', '--config', str(CONFIG_PATH)], check=True)
RUN_DIR = Path(yaml.safe_load(CONFIG_PATH.read_text())['output_dir'])
print((RUN_DIR / 'summary.json').read_text())
print((RUN_DIR / 'best_metrics.json').read_text())

## Development evaluation

This uses only the validation holdout. It does **not** touch final subsets 1 and 4.

In [ ]:
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
subprocess.run([sys.executable, 'eval.py', '--config', str(CONFIG_PATH), '--weights', str(BEST_WEIGHTS), '--split', 'validation', '--class-metrics'], check=True)

## Final evaluation — run only after the model and configuration are locked

Set the guard to `True` only once, for the final submission checkpoint.

In [ ]:
CONFIRM_FINAL_EVALUATION = False
if not CONFIRM_FINAL_EVALUATION:
    raise RuntimeError('Final test evaluation is locked')
subprocess.run([sys.executable, 'eval.py', '--config', str(CONFIG_PATH), '--weights', str(BEST_WEIGHTS), '--split', 'test', '--class-metrics'], check=True)